In [ ]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv



In [ ]:
# 1. 환경 변수 로드
load_dotenv(find_dotenv())
API_KEY = os.getenv('OPINET_API_KEY_1')
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# API 기본 호출 URL
url = "https://www.opinet.co.kr/api/aroundAll.do"
#os 라이브러리를 활용하여, geographic_master_table.parquet의 절대 경로를 찾음.
base_path = os.path.dirname(os.path.abspath("geographic_master_table.parquet"))

# 이 절대경로에다가 ~.parquet 파일을 붙여서 경로를 만듬
master_table_path = os.path.join(base_path, "geographic_master_table.parquet")

master_table = duckdb.read_parquet(master_table_path).df()

In [ ]:
by_gasstation = []
for index, row in master_table.iloc[100:102].iterrows():
    print(f"[{index}] 좌표 ({row['lat']}, {row['lon']}) 검색 중...")
    params = {
        "code": API_KEY,
        "out": "json",
        "x": row["katec_x"],
        "y": row["katec_y"],
        "radius": 5000,
        "prodcd": "B027" , #"B027",  # 휘발유 기준 (경유는 D047)
        "sort": 1          # 1: 가격순, 2: 거리순
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # 에러 발생 시 예외 처리
        data = response.json()
     
        oil_list = data.get('RESULT', {}).get('OIL', [])
        print(f"   -> 검색 결과 {len(oil_list)}개 발견!")

        for station in oil_list:
            by_gasstation.append ((
                row["katec_x"] , 
                row["katec_y"] , 
                station["UNI_ID"],
                station["POLL_DIV_CD"],
                station["OS_NM"],
                station["GIS_X_COOR"],
                station["GIS_Y_COOR"],
                params["prodcd"], # 나중에 함수로 만들때는 이걸 인자로 넣도록 하고, 일단 api호출값에서는 이 prodcd가 없어서 이걸 수기로 테이블에 넣는 방법을 택했다.
                station["PRICE"]
                ))
    except Exception as e:
        print(f"[ERROR] API 호출 중 오류 발생: {e}")
columns =['katec_x' , 'katec_y' , 'uni_cd' , 'brand_cd',
           'station_name' , 'station_x' , 'station_y' , 'fuel_type' , 'price' ]
final_result = pd.DataFrame(by_gasstation , columns=columns)
final_result = final_result.drop_duplicates(subset='uni_cd')
display(final_result)




[100] 좌표 (127.16633, 37.25288) 검색 중...
   -> 검색 결과 31개 발견!
[101] 좌표 (127.26296, 37.25288) 검색 중...
   -> 검색 결과 29개 발견!


,katec_x,katec_y,uni_cd,brand_cd,station_name,station_x,station_y,fuel_type,price
0,326055.650688,517420.407487,A0008815,ETC,오일필드㈜ 기흥지점,324474.68898,517079.72331,B027,1629
1,326055.650688,517420.407487,A0009318,SOL,㈜알찬에너지,324003.56553,517999.84409,B027,1629
2,326055.650688,517420.407487,A0007448,SKE,삼미상사(주)한양주유소,323786.20000,518262.00000,B027,1629
3,326055.650688,517420.407487,A0008820,RTO,㈜셀프제일에너지 용인SELF,324029.00000,517052.00000,B027,1630
4,326055.650688,517420.407487,A0008919,SOL,신동백주유소,323892.80645,520703.41607,B027,1634
5,326055.650688,517420.407487,A0033236,NHO,포곡농협가득채움주유소,328651.53556,519972.37575,B027,1635
6,326055.650688,517420.407487,A0033753,SKE,동백자연에너지,325476.11150,521074.21254,B027,1635
7,326055.650688,517420.407487,A0033534,SOL,청정에너지 동백점,326740.52349,520502.85945,B027,1635
8,326055.650688,517420.407487,A0007507,HDO,㈜서율주유소,323689.00000,519158.00000,B027,1638
9,326055.650688,517420.407487,A0008864,SOL,동백시티주유소,324868.67160,521041.60250,B027,1645
